# Authors Data Generator

Generates JS data modules for the Authors section visualizations.

**Output files** (in `site/data/authors/`):
- `authorMetricsData.js` → authorMetrics.js
- `authorsPerPaperData.js` → authorsPerPaper.js
- `collaborationNetworkData.js` → collaborationNetwork.js
- `uniqueAuthorsData.js` → uniqueAuthorsTimeline.js

In [34]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Paths
DATA = Path("../data/processed/outputs/openalex_notebook_outputs/tables")
OUT = Path("../site/data/authors")
OUT.mkdir(parents=True, exist_ok=True)

print(f"Source: {DATA}")
print(f"Output: {OUT}")

Source: ../data/processed/outputs/openalex_notebook_outputs/tables
Output: ../site/data/authors


---
## 1. Author Metrics (Bubble Chart)

In [35]:
# Load author stats
df = pd.read_csv(DATA / "sec2d_author_stats.csv")

# Categorize authors
def categorize(r):
    if r["papers"] >= 30: return "prolific"
    if r["citations"] >= 3000: return "highly-cited"
    if r["papers"] >= 10: return "steady"
    return "emerging"

df["category"] = df.apply(categorize, axis=1)

# Top 100 authors by papers + citations
df["score"] = df["papers"] + df["citations"] / 100
top = df.nlargest(100, "score")

# Build data
data = top[["author_name", "papers", "citations", "awards", "category"]].copy()
data.columns = ["name", "papers", "citations", "awards", "category"]
data["awards"] = data["awards"].fillna(0).astype(int)
records = data.to_dict(orient="records")

# Stats with categories descriptions (required by JS)
stats = {
    "totalAuthors": len(df),
    "maxPapers": int(df["papers"].max()),
    "maxCitations": int(df["citations"].max()),
    "maxAwards": int(df["awards"].max()),
    "avgPapers": round(df["papers"].mean(), 1),
    "avgCitations": round(df["citations"].mean(), 1),
    "categories": {
        "prolific": "30+ papers",
        "highly-cited": "3000+ citations",
        "steady": "10+ papers",
        "emerging": "New authors"
    }
}

# JS output
js = (
    "/** AUTO-GENERATED */\n"
    f"export const authorMetricsData = {json.dumps(records)};\n\n"
    f"export const authorMetricsStats = {json.dumps(stats)};\n"
)
(OUT / "authorMetricsData.js").write_text(js)
print(f"✓ authorMetricsData.js | {len(records)} authors")

✓ authorMetricsData.js | 100 authors


---
## 2. Authors per Paper (Line Chart)

In [36]:
# Load authors per paper by year
df = pd.read_csv(DATA / "sec2a_authors_per_paper_by_year.csv")

# Build data with error bands
records = []
for _, r in df.iterrows():
    avg = r["avg_authors"]
    var = r.get("std_authors", avg * 0.3)  # fallback variance
    records.append({
        "year": int(r["Year"]),
        "avg": round(avg, 2),
        "variance": round(var, 2),
        "min": round(max(1, avg - var), 2),
        "max": round(avg + var, 2)
    })

# Stats
stats = {
    "overallAvg": round(np.mean([r["avg"] for r in records]), 2),
    "minYear": min(r["year"] for r in records),
    "maxYear": max(r["year"] for r in records),
    "trend": "increasing" if records[-1]["avg"] > records[0]["avg"] else "stable"
}

# JS output
js = (
    "/** AUTO-GENERATED */\n"
    f"export const authorsPerPaperData = {json.dumps(records)};\n\n"
    f"export const authorsPerPaperStats = {json.dumps(stats)};\n"
)
(OUT / "authorsPerPaperData.js").write_text(js)
print(f"✓ authorsPerPaperData.js | {len(records)} years")

✓ authorsPerPaperData.js | 35 years


---
## 3. Unique Authors Timeline (Area Chart)

In [37]:
# Load cumulative unique authors
df = pd.read_csv(DATA / "sec2b_unique_authors_cumulative.csv")

# Build data
records = []
prev = 0
for _, r in df.iterrows():
    cum = int(r["cumulative_unique_authors"])
    records.append({
        "year": int(r["Year"]),
        "cumulative": cum,
        "newAuthors": cum - prev
    })
    prev = cum

# Stats
stats = {
    "total": records[-1]["cumulative"],
    "peakNewYear": max(records, key=lambda x: x["newAuthors"])["year"],
    "peakNewCount": max(r["newAuthors"] for r in records),
    "avgNewPerYear": round(np.mean([r["newAuthors"] for r in records]), 1)
}

# JS output
js = (
    "/** AUTO-GENERATED */\n"
    f"export const uniqueAuthorsData = {json.dumps(records)};\n\n"
    f"export const uniqueAuthorsStats = {json.dumps(stats)};\n"
)
(OUT / "uniqueAuthorsData.js").write_text(js)
print(f"✓ uniqueAuthorsData.js | total: {stats['total']:,} authors")

✓ uniqueAuthorsData.js | total: 7,219 authors


---
## 4. Collaboration Network (Force Graph)

In [38]:
# Load coauthor edges and author stats
edges = pd.read_csv(DATA / "coauthor_edges.csv")
authors = pd.read_csv(DATA / "sec2d_author_stats.csv")

# Top authors by collaborations
TOP_N = 50
collab_counts = pd.concat([edges["author_a"], edges["author_b"]]).value_counts()
top_ids = set(collab_counts.head(TOP_N).index)

# Filter edges between top authors
edges_top = edges[(edges["author_a"].isin(top_ids)) & (edges["author_b"].isin(top_ids))].copy()

# Build nodes
author_map = authors.set_index("author_id").to_dict(orient="index")
nodes = []
for aid in top_ids:
    info = author_map.get(aid, {})
    nodes.append({
        "id": info.get("author_name", aid),
        "group": 1 + hash(aid) % 3,  # simple grouping
        "papers": int(info.get("papers", 0)),
        "collaborations": int(collab_counts.get(aid, 0)),
        "author_id": aid
    })

# Build links
name_map = {a["author_id"]: a["id"] for a in nodes}
links = []
for _, r in edges_top.iterrows():
    src = name_map.get(r["author_a"])
    tgt = name_map.get(r["author_b"])
    if src and tgt:
        links.append({"source": src, "target": tgt, "value": int(r["weight"])})

# Network data
network = {"nodes": nodes, "links": links}

# Stats with groups descriptions (required by JS)
stats = {
    "totalNodes": len(nodes),
    "totalLinks": len(links),
    "avgCollaborations": round(np.mean([n["collaborations"] for n in nodes]), 1),
    "maxCollaborations": max(n["collaborations"] for n in nodes),
    "groups": {
        1: "Visualization",
        2: "Analytics",
        3: "Systems"
    }
}

# JS output
js = (
    "/** AUTO-GENERATED */\n"
    f"export const collaborationNetworkData = {json.dumps(network)};\n\n"
    f"export const networkStats = {json.dumps(stats)};\n"
)
(OUT / "collaborationNetworkData.js").write_text(js)
print(f"✓ collaborationNetworkData.js | {len(nodes)} nodes, {len(links)} links")

✓ collaborationNetworkData.js | 50 nodes, 133 links


---
## Summary

Generated files:
```
site/data/authors/
├── authorMetricsData.js
├── authorsPerPaperData.js
├── collaborationNetworkData.js
└── uniqueAuthorsData.js
```